# Chapter 2: Regression — Predicting Numbers

Regression answers: **"How much?"** or **"How many?"**

- What will the house price be?
- How many units will we sell?
- What temperature will it be tomorrow?

The model learns a **mathematical relationship** between input features and a numeric output.

---
## Linear Regression: Drawing the Best Line

The simplest model: fit a straight line through the data.

```
y = mx + b
```
- **m** = slope (how much y changes per unit of x)
- **b** = intercept (where the line crosses y-axis)

The model finds the m and b that minimize the total error.

In [ ]:
import numpy as np

# Example: house size (sq ft) → price ($)
sizes = np.array([600, 800, 1000, 1200, 1400, 1600, 1800, 2000])
prices = np.array([150000, 180000, 220000, 260000, 290000, 320000, 365000, 400000])

print("House Data:")
print(f"{'Size (sq ft)':>12} {'Price ($)':>12}")
print(f"{'─'*12} {'─'*12}")
for s, p in zip(sizes, prices):
    print(f"{s:>12,} {p:>12,}")

In [ ]:
from sklearn.linear_model import LinearRegression

# Train linear regression
X = sizes.reshape(-1, 1)  # sklearn needs 2D array
model = LinearRegression()
model.fit(X, prices)

slope = model.coef_[0]
intercept = model.intercept_

print("Linear Regression Result:")
print(f"  Formula: price = {slope:.1f} × size + {intercept:,.0f}")
print(f"  Slope:     {slope:.1f} (each extra sq ft adds ${slope:.0f})")
print(f"  Intercept: {intercept:,.0f} (base price)")
print()

# Predict for new sizes
print("Predictions:")
for new_size in [900, 1500, 2500]:
    pred = model.predict([[new_size]])[0]
    print(f"  {new_size:,} sq ft → ${pred:,.0f}")

---
## How Does It Find the Best Line?

It tries to **minimize the total error** (sum of all distances from points to the line).

The error metric is called **Mean Squared Error (MSE)**:
```
MSE = average of (actual - predicted)²
```

Why squared? So positive and negative errors don't cancel out, and big errors get penalized more.

In [ ]:
# Visualize error for different lines
predictions = model.predict(X)

print("How the best line fits each data point:")
print(f"{'Size':>8} {'Actual':>12} {'Predicted':>12} {'Error':>12} {'Error²':>14}")
print(f"{'─'*8} {'─'*12} {'─'*12} {'─'*12} {'─'*14}")

total_sq_error = 0
for s, actual, pred in zip(sizes, prices, predictions):
    error = actual - pred
    sq_error = error ** 2
    total_sq_error += sq_error
    print(f"{s:>8,} {actual:>12,} {pred:>12,.0f} {error:>+12,.0f} {sq_error:>14,.0f}")

mse = total_sq_error / len(sizes)
print(f"\nMean Squared Error (MSE): {mse:,.0f}")
print(f"Root MSE (RMSE): ${np.sqrt(mse):,.0f}")
print(f"\nRMSE means: on average, predictions are off by about ${np.sqrt(mse):,.0f}")

In [ ]:
# Compare good line vs bad lines
print("What if we used different slopes?\n")
print(f"{'Slope':>8} {'Intercept':>12} {'MSE':>16} {'Quality':>10}")
print(f"{'─'*8} {'─'*12} {'─'*16} {'─'*10}")

test_slopes = [100, 150, slope, 200, 250]
for s in test_slopes:
    preds = s * sizes + intercept
    mse_test = np.mean((prices - preds) ** 2)
    quality = "← BEST" if s == slope else ""
    print(f"{s:>8.1f} {intercept:>12,.0f} {mse_test:>16,.0f} {quality:>10}")

print("\nThe model found the slope that gives the LOWEST MSE.")

---
## Multiple Features

Real-world predictions use **multiple inputs**:

```
price = w1×size + w2×bedrooms + w3×age + b
```

Each weight (w) tells us how important that feature is.

In [ ]:
# Multiple feature regression
# [size, bedrooms, age_years]
houses = np.array([
    [600,  1, 30],
    [800,  2, 20],
    [1000, 2, 15],
    [1200, 3, 10],
    [1400, 3,  5],
    [1600, 4,  2],
    [1800, 4,  1],
    [2000, 5,  0],
])
prices_multi = np.array([150000, 190000, 230000, 280000, 310000, 350000, 380000, 420000])

model_multi = LinearRegression()
model_multi.fit(houses, prices_multi)

feature_names = ["size (sq ft)", "bedrooms", "age (years)"]
print("Multiple Feature Regression:")
print(f"  price = ", end="")
parts = []
for name, coef in zip(feature_names, model_multi.coef_):
    parts.append(f"{coef:+,.1f} × {name}")
print(" + ".join(parts))
print(f"         + {model_multi.intercept_:,.0f}")

print("\nWhat each weight means:")
for name, coef in zip(feature_names, model_multi.coef_):
    direction = "adds" if coef > 0 else "subtracts"
    print(f"  Each unit of {name}: {direction} ${abs(coef):,.0f}")

# Predict
print("\nPredictions:")
test = [[1100, 3, 8], [1500, 2, 20], [2200, 5, 0]]
for t in test:
    pred = model_multi.predict([t])[0]
    print(f"  {t[0]} sqft, {t[1]} bed, {t[2]}yr old → ${pred:,.0f}")

---
## Polynomial Regression: When Lines Aren't Enough

Some relationships are **curved**, not straight.

```
Linear:     y = mx + b           (straight line)
Polynomial: y = ax² + bx + c     (curve)
```

Example: a car's fuel efficiency doesn't increase linearly with speed — it peaks around 55mph and drops at higher speeds.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# Curved data: speed → fuel efficiency (mpg)
np.random.seed(42)
speed = np.array([20, 30, 40, 50, 55, 60, 65, 70, 80, 90, 100])
mpg = np.array([22, 28, 33, 36, 37, 36, 34, 31, 25, 18, 12])
mpg = mpg + np.random.normal(0, 1, len(mpg))  # add some noise

X_speed = speed.reshape(-1, 1)

# Linear model
linear = LinearRegression()
linear.fit(X_speed, mpg)
pred_linear = linear.predict(X_speed)

# Polynomial model (degree 2 = quadratic curve)
poly = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly.fit(X_speed, mpg)
pred_poly = poly.predict(X_speed)

mse_linear = mean_squared_error(mpg, pred_linear)
mse_poly = mean_squared_error(mpg, pred_poly)

print("Speed → Fuel Efficiency (MPG)")
print(f"{'Speed':>6} {'Actual':>8} {'Linear':>8} {'Poly(2)':>8}")
print(f"{'─'*6} {'─'*8} {'─'*8} {'─'*8}")
for s, a, pl, pp in zip(speed, mpg, pred_linear, pred_poly):
    print(f"{s:>6} {a:>8.1f} {pl:>8.1f} {pp:>8.1f}")

print(f"\nLinear MSE:     {mse_linear:.2f}")
print(f"Polynomial MSE: {mse_poly:.2f}")
print(f"\nPolynomial fits {mse_linear/mse_poly:.1f}x better!")
print("Because the real relationship is curved, not straight.")

---
## Danger: Too Much Polynomial = Overfitting

Higher degree polynomials can fit training data perfectly but fail on new data.

In [ ]:
# Compare polynomial degrees
from sklearn.model_selection import cross_val_score

print("Polynomial Degree vs Error:")
print(f"{'Degree':>8} {'Train MSE':>12} {'Behavior':>20}")
print(f"{'─'*8} {'─'*12} {'─'*20}")

for degree in [1, 2, 3, 5, 8]:
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X_speed, mpg)
    pred = model.predict(X_speed)
    mse = mean_squared_error(mpg, pred)
    
    if degree == 1:
        behavior = "too simple (underfit)"
    elif degree == 2:
        behavior = "← just right"
    elif degree <= 3:
        behavior = "good"
    else:
        behavior = "too complex (overfit)"
    
    print(f"{degree:>8} {mse:>12.2f} {behavior:>20}")

print("\nLower training error ≠ better model!")
print("Degree 8 memorizes noise, degree 2 captures the real pattern.")

---
## R² Score: How Good Is the Fit?

R² (R-squared) tells us what percentage of the data's variation our model explains.

- **R² = 1.0**: perfect prediction
- **R² = 0.0**: no better than predicting the average every time
- **R² < 0**: worse than the average (bad model)

In [ ]:
from sklearn.metrics import r2_score

# Back to house prices
pred_simple = model.predict(sizes.reshape(-1, 1))
r2 = r2_score(prices, pred_simple)

print(f"House Price Model:")
print(f"  R² = {r2:.4f}")
print(f"  This means: {r2:.1%} of price variation is explained by size")
print()

# Show what R² values mean
print("R² Interpretation Guide:")
ranges = [(0.9, 1.0, "Excellent"), (0.7, 0.9, "Good"), 
          (0.5, 0.7, "Moderate"), (0.0, 0.5, "Weak")]
for low, high, label in ranges:
    marker = " ← our model" if low <= r2 <= high else ""
    print(f"  {low:.1f} - {high:.1f}: {label}{marker}")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Regression** | Predict a continuous number |
| **Linear** | `y = mx + b` — best for straight-line relationships |
| **Polynomial** | `y = ax² + bx + c` — captures curves |
| **MSE** | Average squared error — lower is better |
| **R²** | Percentage of variation explained — closer to 1 is better |
| **Multiple features** | Use many inputs, each with its own weight |
| **Overfitting** | Too complex = memorizes noise, fails on new data |

**Next chapter**: Classification — predicting categories instead of numbers